# Model-based best drive time map

Recreates the Folium **raster heatmap** from [`03-osrm_access.ipynb`](03-osrm_access.ipynb) (`osrm_best_drive_time_map.html`), but **best drive time** at each address is the minimum over pharmacies of the **linear surrogate** from [`06-distill.ipynb`](06-distill.ipynb) ([`../data/drive_time_surrogate.json`](../data/drive_time_surrogate.json)).

Interpolation from address points to a grid, county mask, and color scale match notebook `03` for easy visual comparison.

In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import numpy as np
import pandas as pd

DATA_DIR = Path("../data")
ADDRESSES_CSV = DATA_DIR / "addresses_with_population_weights.csv"
PHARMACIES_JSON = DATA_DIR / "pharmacies_active.json"
SURROGATE_JSON = DATA_DIR / "drive_time_surrogate.json"
PHARM_CHUNK = 32


def haversine_km(lat1: np.ndarray, lon1: np.ndarray, lat2: np.ndarray, lon2: np.ndarray) -> np.ndarray:
    r = 6371.0
    p1 = np.radians(lat1)
    p2 = np.radians(lat2)
    dp = np.radians(lat2 - lat1)
    dl = np.radians(lon2 - lon1)
    a = np.sin(dp / 2) ** 2 + np.cos(p1) * np.cos(p2) * np.sin(dl / 2) ** 2
    return (2 * r * np.arcsin(np.sqrt(np.clip(a, 0.0, 1.0)))).astype(np.float64)


def bearing_sin_cos(
    lat1: np.ndarray, lon1: np.ndarray, lat2: np.ndarray, lon2: np.ndarray
) -> tuple[np.ndarray, np.ndarray]:
    p1 = np.radians(lat1)
    p2 = np.radians(lat2)
    dl = np.radians(lon2 - lon1)
    x = np.sin(dl) * np.cos(p2)
    y = np.cos(p1) * np.sin(p2) - np.sin(p1) * np.cos(p2) * np.cos(dl)
    theta = np.arctan2(x, y)
    sin_b, cos_b = np.sin(theta), np.cos(theta)
    degenerate = (np.hypot(x, y) < 1e-15) | ~np.isfinite(theta)
    sin_b = np.where(degenerate, 0.0, sin_b)
    cos_b = np.where(degenerate, 1.0, cos_b)
    return sin_b.astype(np.float64), cos_b.astype(np.float64)


def min_modeled_drive_seconds(
    lat_a: np.ndarray,
    lon_a: np.ndarray,
    lat_p: np.ndarray,
    lon_p: np.ndarray,
    model: dict,
    chunk: int = PHARM_CHUNK,
) -> np.ndarray:
    """Per address: min over pharmacies of surrogate seconds (≥ 0), same forward pass as `driveTimeSurrogate.ts`."""
    bbox = model["bbox"]
    lat_min, lat_max = float(bbox["lat_min"]), float(bbox["lat_max"])
    lon_min, lon_max = float(bbox["lon_min"]), float(bbox["lon_max"])
    dlat = lat_max - lat_min
    dlon = lon_max - lon_min
    if dlat <= 0:
        dlat = 1.0
    if dlon <= 0:
        dlon = 1.0

    order = list(model["feature_order"])
    coef = np.asarray(model["coef"], dtype=np.float64)
    mean = np.asarray(model["scaler_mean"], dtype=np.float64)
    scale = np.asarray(model["scaler_scale"], dtype=np.float64)
    intercept = float(model["intercept"])
    if coef.shape != mean.shape or coef.shape != scale.shape:
        raise ValueError("coef / scaler length mismatch")
    if len(order) != coef.shape[0]:
        raise ValueError("feature_order length does not match coef")

    n = lat_a.shape[0]
    p = lat_p.shape[0]
    lat_a_c = lat_a[:, np.newaxis]
    lon_a_c = lon_a[:, np.newaxis]
    min_sec = np.full(n, np.inf, dtype=np.float64)

    for j0 in range(0, p, chunk):
        j1 = min(p, j0 + chunk)
        lp = lat_p[j0:j1][np.newaxis, :]
        lop = lon_p[j0:j1][np.newaxis, :]
        hav = haversine_km(lat_a_c, lon_a_c, lp, lop)
        sin_b, cos_b = bearing_sin_cos(lat_a_c, lon_a_c, lp, lop)
        sh = hav.shape
        layers: dict[str, np.ndarray] = {
            "norm_lat": np.broadcast_to((lat_a_c - lat_min) / dlat, sh),
            "norm_lon": np.broadcast_to((lon_a_c - lon_min) / dlon, sh),
            "norm_pharm_lat": np.broadcast_to((lp - lat_min) / dlat, sh),
            "norm_pharm_lon": np.broadcast_to((lop - lon_min) / dlon, sh),
            "haversine_km": hav,
            "log1p_haversine_km": np.log1p(hav),
            "sin_bearing": sin_b,
            "cos_bearing": cos_b,
        }
        try:
            stacks = [layers[name] for name in order]
        except KeyError as e:
            raise KeyError(f"unknown feature in JSON: {e}; supported: {sorted(layers)}") from e
        x = np.stack(stacks, axis=-1)
        z = (x - mean) / scale
        pred = intercept + np.dot(z, coef)
        pred = np.maximum(0.0, pred)
        min_sec = np.minimum(min_sec, pred.min(axis=1))

    return min_sec


with SURROGATE_JSON.open(encoding="utf-8") as f:
    surrogate = json.load(f)

addresses_df = pd.read_csv(ADDRESSES_CSV)
with PHARMACIES_JSON.open(encoding="utf-8") as f:
    pharm_records = json.load(f)
plat = np.array([r["lat"] for r in pharm_records], dtype=np.float64)
plon = np.array([r["lon"] for r in pharm_records], dtype=np.float64)

lat_a = addresses_df["lat"].to_numpy(dtype=np.float64)
lon_a = addresses_df["lon"].to_numpy(dtype=np.float64)
min_sec = min_modeled_drive_seconds(lat_a, lon_a, plat, plon, surrogate)
drive_times_df = addresses_df.assign(min_drive_minutes=min_sec / 60.0)

print(
    f"surrogate schema_version={surrogate.get('schema_version')} "
    f"features={surrogate['feature_order']}"
)
print(f"addresses {len(drive_times_df):,}; pharmacies {len(plat):,}")
print(
    "modeled best drive (min): {:.1f} min; median: {:.1f}; max: {:.1f}".format(
        float(drive_times_df["min_drive_minutes"].min()),
        float(drive_times_df["min_drive_minutes"].median()),
        float(drive_times_df["min_drive_minutes"].max()),
    )
)

In [ ]:
import branca.colormap as cm
import folium
import geopandas as gpd
import matplotlib.colors as mcolors
import numpy as np
import shapely
from scipy.interpolate import griddata
from shapely.ops import unary_union

FIVE_COUNTIES_GEOJSON = DATA_DIR / "counties_five_vt.geojson"
LEGACY_COUNTIES_GEOJSON = DATA_DIR / "counties.geojson"
LEGACY_REGION_GEOJSON = DATA_DIR / "target_region.geojson"
MAP_HTML = DATA_DIR / "model_best_drive_time_map.html"

pharmacies_df = pd.DataFrame(pharm_records)

if FIVE_COUNTIES_GEOJSON.exists():
    boundary_gdf = gpd.read_file(FIVE_COUNTIES_GEOJSON)
    boundary_name = "County boundaries"
elif LEGACY_COUNTIES_GEOJSON.exists():
    boundary_gdf = gpd.read_file(LEGACY_COUNTIES_GEOJSON)
    boundary_name = "County boundaries"
else:
    boundary_gdf = gpd.read_file(LEGACY_REGION_GEOJSON)
    boundary_name = "Study region"

region = unary_union(boundary_gdf.geometry)
minx, miny, maxx, maxy = region.bounds

GRID_RES = 300
lon_grid = np.linspace(minx, maxx, GRID_RES)
lat_grid = np.linspace(miny, maxy, GRID_RES)
lon_mesh, lat_mesh = np.meshgrid(lon_grid, lat_grid)

pts = drive_times_df[["lon", "lat"]].to_numpy()
vals = drive_times_df["min_drive_minutes"].to_numpy()

grid_z = griddata(pts, vals, (lon_mesh, lat_mesh), method="linear")
grid_nn = griddata(pts, vals, (lon_mesh, lat_mesh), method="nearest")
grid_z = np.where(np.isnan(grid_z), grid_nn, grid_z)
grid_z = np.clip(grid_z, 0, None)

grid_pts = shapely.points(lon_mesh.ravel(), lat_mesh.ravel())
inside = shapely.within(grid_pts, region).reshape(lon_mesh.shape)
grid_z[~inside] = np.nan

CMAP_COLORS = ["#2ecc71", "#82e0aa", "#f9e79f", "#f0b27a", "#e74c3c", "#8b0000"]
CMAP_BREAKS = np.array([0, 10, 15, 20, 30, 45, 60], dtype=float)
CMAP_TARGETS = np.linspace(0, 1, len(CMAP_BREAKS))

_crgba = [mcolors.to_rgba(c) for c in CMAP_COLORS]
_pos = np.linspace(0, 1, len(CMAP_COLORS))
_cdict = {
    ch: [(_pos[i], _crgba[i][j], _crgba[i][j]) for i in range(len(_crgba))]
    for j, ch in enumerate(["red", "green", "blue"])
}
drive_cmap = mcolors.LinearSegmentedColormap("drive_time", _cdict)

normed = np.interp(np.clip(grid_z, 0, 60), CMAP_BREAKS, CMAP_TARGETS)
rgba = drive_cmap(normed)
rgba[..., 3] = np.where(np.isnan(grid_z), 0.0, 0.45)
rgba = np.flipud(rgba)

center_lat = drive_times_df["lat"].mean()
center_lon = drive_times_df["lon"].mean()

m = folium.Map(location=[center_lat, center_lon], zoom_start=9, tiles="cartodbpositron")

boundary_kwargs: dict = {
    "name": boundary_name,
    "style_function": lambda _: {"fillOpacity": 0, "color": "#333333", "weight": 2},
}
if "county_name" in boundary_gdf.columns:
    boundary_kwargs["tooltip"] = folium.GeoJsonTooltip(
        fields=["county_name"], aliases=["County"]
    )
folium.GeoJson(boundary_gdf.__geo_interface__, **boundary_kwargs).add_to(m)

folium.raster_layers.ImageOverlay(
    image=rgba,
    opacity=0.6,
    bounds=[[miny, minx], [maxy, maxx]],
    name="Best drive time (model)",
    interactive=False,
    zindex=1,
).add_to(m)

pharm_layer = folium.FeatureGroup(name="Pharmacies")
for _, row in pharmacies_df.iterrows():
    folium.CircleMarker(
        location=[row["lat"], row["lon"]],
        radius=6,
        color="#2c3e50",
        fill=True,
        fill_color="#3498db",
        fill_opacity=0.9,
        weight=1,
        tooltip=f"{row['name']} ({row['city']})",
    ).add_to(pharm_layer)
pharm_layer.add_to(m)

colormap = cm.LinearColormap(
    colors=CMAP_COLORS + ["#8b0000"],
    index=[0, 10, 15, 20, 30, 45, 60],
    vmin=0,
    vmax=60,
    caption="Best drive time to any pharmacy (minutes, linear surrogate)",
)
colormap.add_to(m)

folium.LayerControl().add_to(m)
m.save(str(MAP_HTML))
print(f"Saved {MAP_HTML}")
m